In [15]:
#Install Libraries
!pip install -q transformers datasets evaluate torchaudio accelerate jiwer

In [16]:
from datasets import load_from_disk, Audio
from transformers import AutoProcessor, AutoModelForCTC, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [17]:
#Load Dataset
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
zip_path = "/content/drive/MyDrive/Colab Notebooks/Project 3 (Voice Recognition)/minds14_augmented.zip"

In [26]:
extract_folder = "/content/minds14_augmented"

In [27]:
#Exstract zip
import zipfile
import os

if not os.path.exists(extract_folder):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      zip_ref.extractall(extract_folder)

      print(f"Zip file '{zip_path}' has been extracted to '{extract_folder}'.")

Zip file '/content/drive/MyDrive/Colab Notebooks/Project 3 (Voice Recognition)/minds14_augmented.zip' has been extracted to '/content/minds14_augmented'.


In [28]:
# Path dataset yang sudah kamu simpan
dataset_path = "/content/minds14_augmented"
dataset_path

'/content/minds14_augmented'

In [29]:
print("📥 Loading dataset...")
dataset = load_from_disk(dataset_path)
dataset

📥 Loading dataset...


DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

In [65]:
#Ambil hanya data train
train_dataset = dataset["train"]
val_dataset = dataset["validation"]

In [66]:
# Hanya ambil kolom audio dan transcription
train_asr = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ['audio', 'transcription']]
)
val_asr = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in ['audio', 'transcription']]
)

In [68]:
# ============================================
# 4. RESAMPLE KE 16kHz (WAJIB!)
# ============================================
TARGET_SR = 16000
print(f"\n🎵 Resampling train data to {TARGET_SR} kHz...")
train_asr = train_asr.cast_column("audio", Audio(sampling_rate=TARGET_SR))
val_asr = val_asr.cast_column("audio", Audio(sampling_rate=TARGET_SR))


🎵 Resampling train data to 16000 kHz...


In [69]:
# ============================================
# 5. NORMALISASI TEKS (UPPERCASE)
# ============================================
print("\n Normalizing transcriptions to uppercase...")

def uppercase_text(example):
    return {"transcription": example["transcription"].upper()}

train_asr = train_asr.map(uppercase_text)
val_asr = val_asr.map(uppercase_text)

# Cek sample
print(f"Sample transcription: {train_asr[0]['transcription'][:100]}...")


 Normalizing transcriptions to uppercase...


Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Sample transcription: BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE...


In [59]:
train_asr.column_names

['input_values', 'labels']

In [70]:
# ============================================
# 6. LOAD PROCESSOR
# ============================================
print("\nLoading Wav2Vec2 processor...")
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
print(f"Processor loaded. Sampling rate: {processor.feature_extractor.sampling_rate} Hz")



Loading Wav2Vec2 processor...
Processor loaded. Sampling rate: 16000 Hz


In [73]:
# ============================================
# 7. PREPROCESSING FUNCTION UNTUK TRAIN DATA
# ============================================
def prepare_dataset(batch):
    """
    Preprocessing untuk ASR:
    1. Extract audio array
    2. Process audio ke input_values
    3. Tokenize transcription ke labels
    """
    audio = batch["audio"]

    # Process audio ke input_values, ensuring it's a PyTorch tensor
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt" # Specify to return PyTorch tensors
    ).input_values[0] # Get the first (and only) item from the list of tensors

    # Tokenize transcription ke labels
    # Use processor.tokenizer directly for text labels
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids

    return batch

print("\n Applying preprocessing to train data...")

# Check if 'audio' column exists before mapping to prevent KeyError on re-execution
if "audio" in train_asr.column_names:
    train_asr = train_asr.map(
        prepare_dataset,
        remove_columns=train_asr.column_names,
        num_proc=4  # Parallel processing
    )
    print(f" Train data preprocessed: {len(train_asr)} samples")
    print(f"   Sample input_values shape: {train_asr[0]['input_values'].shape}")
    print(f"   Sample labels shape: {len(train_asr[0]['labels'])}")
else:
    print("Skipping preprocessing for train data: 'audio' column not found (already processed?).")

# Check if 'audio' column exists before mapping to prevent KeyError on re-execution
if "audio" in val_asr.column_names:
    val_asr = val_asr.map(
        prepare_dataset,
        remove_columns=val_asr.column_names,
        num_proc=4  # Parallel processing
    )
    print(f" val data preprocessed: {len(val_asr)} samples")
    print(f"   Sample input_values shape: {val_asr[0]['input_values'].shape}")
    print(f"   Sample labels shape: {len(val_asr[0]['labels'])}")
else:
    print("Skipping preprocessing for train data: 'audio' column not found (already processed?).")


 Applying preprocessing to train data...
Skipping preprocessing for train data: 'audio' column not found (already processed?).
Skipping preprocessing for train data: 'audio' column not found (already processed?).


In [74]:
# ============================================
# 9. LOAD METRIK WER UNTUK EVALUASI
# ============================================
print("\n📊 Loading WER metric...")
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Menghitung WER untuk evaluasi model
    """
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # Ganti -100 dengan pad_token_id untuk decoding
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions dan labels ke string
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Hitung WER
    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}



📊 Loading WER metric...


In [75]:
# ============================================
# 10. LOAD MODEL WAV2VEC2 BASE
# ============================================
print("\n🤖 Loading Wav2Vec2 Base model...")
model = AutoModelForCTC.from_pretrained(
    "facebook/wav2vec2-base",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

print(f"✅ Model loaded. Vocabulary size: {model.config.vocab_size}")
print(f"   Model parameters: {model.num_parameters():,}")


🤖 Loading Wav2Vec2 Base model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded. Vocabulary size: 32
   Model parameters: 94,396,320


In [83]:
# ============================================
# 11. TRAINING ARGUMENTS (DENGAN EVALUASI)
# ============================================
training_args = TrainingArguments(
    output_dir="./wav2vec2-minds14-asr",
    per_device_train_batch_size=4, # Reduced batch size to mitigate OOM error
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="steps",      # Evaluasi berkala
    eval_steps=200,                    # Evaluasi setiap 200 steps
    logging_steps=100,
    save_steps=200,                    # Fixed: Changed to be a multiple of eval_steps (200)
    num_train_epochs=3, # Reduced for faster testing after OOM
    learning_rate=3e-5,
    warmup_steps=500,
    fp16=True,
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,       # Load model terbaik berdasarkan eval loss
    metric_for_best_model="wer",       # Gunakan WER untuk menentukan model terbaik
    greater_is_better=False,           # WER lebih kecil lebih baik
    report_to="none",
)

In [84]:
# ============================================
# 12. INITIALIZE TRAINER (DENGAN VALIDATION)
# ============================================
print("\n🏋️ Initializing Trainer with validation set...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_asr,
    eval_dataset=val_asr,              # Validation dataset dari split awal
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


🏋️ Initializing Trainer with validation set...


In [ ]:
# ============================================
# 13. START TRAINING
# ============================================
print("\n🚀 Starting training with validation monitoring...")
print("="*50)
trainer.train()


🚀 Starting training with validation monitoring...


Step,Training Loss,Validation Loss,Wer
200,2.166950,-0.064904,1.000000
400,0.832995,-0.649503,1.000000
600,0.224666,-0.757390,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=675, training_loss=3.865147543306704, metrics={'train_runtime': 1147.0041, 'train_samples_per_second': 4.708, 'train_steps_per_second': 0.588, 'total_flos': 7.659802511264333e+17, 'train_loss': 3.865147543306704, 'epoch': 3.0})